# Bismillah 
- Setting up My WorkSpace
- And Importing Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# 1. main eik sath he data sets ko load kar raha hun alag alag
# variable ka use krte hue

customers = pd.read_excel('customers.xlsx')
geography = pd.read_excel('geography.xlsx')
products = pd.read_excel('products.xlsx')
reviews = pd.read_excel('customer_reviews.xlsx')
engagement = pd.read_excel('engagement_data.xlsx')
journey = pd.read_excel('customer_journey.xlsx')

# Step 2: Clean the 'Engagement Data' (Crucial Step!)

In [3]:
# Create a copy to clean
engagement_clean = engagement.copy()

In [4]:
engagement_clean.head(20)

,EngagementID,ContentID,ContentType,Likes,EngagementDate,CompaignID,ProductID,ViewsClickCombined
0,1,39,Blog,190,2023-08-30,1,9,1883-671
1,2,48,Blog,114,2023-03-28,18,20,5280-532
2,3,16,video,32,2023-12-08,7,14,1905-204
3,4,43,Video,17,2025-01-21,19,20,2766-257
4,5,16,newsletter,306,2024-02-21,6,15,5116-1524
5,6,32,Socialmedia,648,2023-06-18,18,19,8237-1641
6,7,33,SOCIALMEDIA,1,2025-10-01,12,2,750-34
7,8,47,Blog,1,2025-03-31,17,6,891-35
8,9,48,blog,123,2024-03-19,13,16,5571-1527
9,10,4,Blog,25,2023-12-03,15,15,4279-297


In [5]:
engagement_clean['ViewsClickCombined'].value_counts

<bound method IndexOpsMixin.value_counts of 0        1883-671
1        5280-532
2        1905-204
3        2766-257
4       5116-1524
          ...    
4618    8115-1872
4619     1273-516
4620     1609-599
4621     3052-374
4622     2458-457
Name: ViewsClickCombined, Length: 4623, dtype: object>

### Remove the two error rows containing date-like entries (e.g., '2000-10-01 00:00:00')

In [6]:
engagement_clean = engagement_clean[~engagement_clean['ViewsClickCombined'].astype(str).str.contains(':')]

### Split the column by the hyphen '~'

In [7]:
split_data = engagement_clean['ViewsClickCombined'].str.split('-', expand=True)
engagement_clean['Views'] = split_data[0].astype(int)
engagement_clean['Clicks'] = split_data[1].astype(int)

### Now engagement_clean has beautiful, numeric 'Views' and 'Clicks' columns!

In [8]:
print(engagement_clean[['Views', 'Clicks']].head())

   Views  Clicks
0   1883     671
1   5280     532
2   1905     204
3   2766     257
4   5116    1524


In [9]:
engagement_clean.head()

,EngagementID,ContentID,ContentType,Likes,EngagementDate,CompaignID,ProductID,ViewsClickCombined,Views,Clicks
0,1,39,Blog,190,2023-08-30,1,9,1883-671,1883,671
1,2,48,Blog,114,2023-03-28,18,20,5280-532,5280,532
2,3,16,video,32,2023-12-08,7,14,1905-204,1905,204
3,4,43,Video,17,2025-01-21,19,20,2766-257,2766,257
4,5,16,newsletter,306,2024-02-21,6,15,5116-1524,5116,1524


# Step 3: Understand How the Files Connect (Data Model)
- Customers & Geography: Connect customers and geography using GeographyID from customers and ID from geography.

- Reviews & Customers/Products: Connect reviews with customers using CustomerID, and with products using ProductID.

- Journey & Customers/Products: Connect journey with customers on CustomerID, and with products on ProductId (note the lowercase 'd'!).

In [11]:
customers_geo = pd.merge(
    customers, 
    geography, 
    left_on='GeographyID', 
    right_on='ID', 
    how='left')
customers_geo = customers_geo.drop(columns=['ID'])

In [13]:
full_reviews = pd.merge(
    reviews, 
    customers_geo, 
    on='CustomerID', 
    how='left')
full_reviews = pd.merge(
    full_reviews, 
    products, 
    on='ProductID', 
    how='left')

In [14]:
print("✅ Saara data clean aur merge ho chuka hai! Ab hum Questions start kar sakte hain.")

✅ Saara data clean aur merge ho chuka hai! Ab hum Questions start kar sakte hain.


# Let's Do the First Question
- Total kitne unique customers hain?

In [15]:
# Question 1
total_unique_customers = customers['CustomerID'].nunique()
print(f"Total Unique Customers: {total_unique_customers}")

Total Unique Customers: 100


In [16]:
# Question 2
# Age bins aur labels define karte hain
bins = [0, 30, 55, 100]
labels = ['Young', 'Adult', 'Senior']

# Naya column 'AgeCategory' banate hain
customers_geo['AgeCategory'] = pd.cut(customers_geo['Age'], bins=bins, labels=labels)

# Distribution check karte hain
age_distribution = customers_geo['AgeCategory'].value_counts()
print("Age Category Distribution:")
print(age_distribution)

Age Category Distribution:
AgeCategory
Adult     55
Young     24
Senior    21
Name: count, dtype: int64


In [17]:
# Question 3
gender_distribution = customers_geo['Gender'].value_counts()
print("Gender Distribution:")
print(gender_distribution)

Gender Distribution:
Gender
Female    54
Male      46
Name: count, dtype: int64


In [19]:
# Question 4
avg_rating_age = full_reviews.groupby('AgeCategory')['Rating'].mean()
print("Average Rating by Age Category:")
print(avg_rating_age)

KeyError: 'AgeCategory'

In [20]:
# Question 5
avg_purchase_gender = full_reviews.groupby('Gender')['Price'].mean()
print("Average Purchase Value by Gender:")
print(avg_purchase_gender)

Average Purchase Value by Gender:
Gender
Female    199.548996
Male      202.839553
Name: Price, dtype: float64


In [21]:
# Question 6
customer_country_counts = customers_geo['Region'].value_counts()
print("Customers per Country:")
print(customer_country_counts)
print(f"\n🏆 Sabse zyada customers is country mein hain: {customer_country_counts.index[0]}")

Customers per Country:
Region
Spain          18
Italy          12
Germany        11
Austria        10
UK             10
Netherlands     9
Belgium         9
Sweden          8
Switzerland     8
France          5
Name: count, dtype: int64

🏆 Sabse zyada customers is country mein hain: Spain


In [22]:
# Question 7
avg_rating_country = full_reviews.groupby('Region')['Rating'].mean()
print("Average Review Rating by Country:")
print(avg_rating_country)

Average Review Rating by Country:
Region
Austria        3.654867
Belgium        3.584615
France         3.754098
Germany        3.744681
Italy          3.675497
Netherlands    3.687943
Spain          3.774319
Sweden         3.689655
Switzerland    3.490741
UK             3.717241
Name: Rating, dtype: float64
